# meeting_extracted_pm_pack — PM 月度審核包（總覽 ＋ 對照結果 ＋ 當月會議摘要；B 節摘要用 LLM）

給 PM 從高層看「彙整結果哪裡要改」，印成 HTML（瀏覽器開，純閱讀）或 markdown，使用者整段複製：

- **總覽卡**：每月每 app 一行數字（會議條目、客戶數、對照格、衝突 / 補充 / 一致…）＋ 固定請 PM 回答的三題。
- **A 節：會議 × Sales Brief 對照結果**：每 app 一張表，**預設只展開衝突與補充**，其餘折疊。資料來自 `v_meeting_sb_reconcile_q`，不重算。
- **B 節：當月會議紀錄彙整**：每個客戶 × 應用一張卡：輪廓（條目 / 信 / 日期 / 主題）、**LLM 寫的 3–5 句摘要 ＋ 值得注意**、規則挑的代表句（要求 / 決定 / 問題優先）；逐條全列折疊在最後。

- 只讀 View / 表，不寫表。LLM 只用在 B 節摘要（一組一次；`llm_summary` / `max_llm_groups` / `min_items_for_llm` 閘門；每月幾十次、USD < 1）。
- summary 已 token 化、無人名；LLM 輸出含 token / email 整組作廢。`quote` 預設不印。
- 參數見 [c01]；兩種「月」：A 節是 `brief_month`；B 節預設按會議日期（`month_basis = meeting`），信件層漏斗永遠按收信日。
- 純函式 [c02]～[c06]、[c08] 由 `tests/test_meeting_extracted_pm_pack.py` 測。


In [ ]:
# [c01] params
# PM 月度資料包：只讀 View / 表，不寫任何表。B 節客戶摘要可呼叫 LLM（llm_summary；受 max_llm_groups 閘門）。由 job parameters 覆蓋；改預設值要先 dbutils.widgets.removeAll()。
import re as _re
from datetime import date as _date

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")
dbutils.widgets.text("months", "")                                      # yyyy-MM，逗號分隔可多月；空 = 本月
dbutils.widgets.dropdown("sections", "A,B", ["A,B", "A", "B"])          # A = 會議 × Sales Brief 對照結果；B = 當月會議紀錄彙整
dbutils.widgets.dropdown("month_basis", "meeting", ["meeting", "mail"])  # B 節條目歸哪個月：會議日期（event_date）/ 收信日期（mail_date）
dbutils.widgets.dropdown("include_dups", "false", ["true", "false"])    # B 節是否列同文重複（鏡頭表 dup_key 非空）的條目
dbutils.widgets.dropdown("include_quote", "false", ["true", "false"])   # B 節全列是否附原文 quote（正文第三人名未遮，只給 PM 內部看才開）
dbutils.widgets.text("max_text", "150")                                 # A 節每格填報 / 會議摘要最多幾字
dbutils.widgets.text("max_per_group", "0")                              # B 節全列每個客戶 × 應用最多列幾條；0 = 不限
dbutils.widgets.text("max_rows", "5000")                                # B 節單月 collect 上限（保護 driver）
dbutils.widgets.dropdown("output", "html", ["html", "md", "both"])      # html：給 PM 瀏覽器開；md：純文字
dbutils.widgets.dropdown("llm_summary", "true", ["true", "false"])      # B 節每個客戶 × 應用請 LLM 寫 3–5 句摘要（false：只有規則挑的代表句）
dbutils.widgets.text("max_llm_groups", "0")                             # 費用閘門：每月最多幾組呼叫 LLM（條目多的先）；0 = 不限
dbutils.widgets.text("min_items_for_llm", "3")                          # 條目少於這個數的組不呼叫 LLM，只列代表句
dbutils.widgets.text("model", "claude-sonnet-5")                        # Azure Foundry 上的模型名（同 T4 / Stage A）
dbutils.widgets.text("concurrency", "4")
dbutils.widgets.dropdown("prompt_cache", "true", ["true", "false"])


def _months(s: str) -> list[str]:
    out = sorted({m.strip() for m in s.split(",") if m.strip()}) or [_date.today().strftime("%Y-%m")]
    for m in out:
        assert _re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", m), f"months 格式 yyyy-MM：{m}"
    return out


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "months": _months(dbutils.widgets.get("months")),
    "sections": [s.strip() for s in dbutils.widgets.get("sections").split(",") if s.strip()],
    "month_basis": dbutils.widgets.get("month_basis"),
    "include_dups": dbutils.widgets.get("include_dups") == "true",
    "include_quote": dbutils.widgets.get("include_quote") == "true",
    "max_text": int(dbutils.widgets.get("max_text")),
    "max_per_group": int(dbutils.widgets.get("max_per_group")),
    "max_rows": int(dbutils.widgets.get("max_rows")),
    "output": dbutils.widgets.get("output"),
    "llm_summary": dbutils.widgets.get("llm_summary") == "true",
    "max_llm_groups": int(dbutils.widgets.get("max_llm_groups")),
    "min_items_for_llm": int(dbutils.widgets.get("min_items_for_llm")),
    "model": dbutils.widgets.get("model").strip(),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "prompt_cache": dbutils.widgets.get("prompt_cache") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert cfg["sections"] and set(cfg["sections"]) <= {"A", "B"}, "sections 只能是 A / B"
assert cfg["month_basis"] in ("meeting", "mail")
assert 50 <= cfg["max_text"] <= 2000 and 0 <= cfg["max_per_group"] <= 5000 and 100 <= cfg["max_rows"] <= 50000
assert cfg["max_llm_groups"] >= 0 and cfg["min_items_for_llm"] >= 1 and cfg["concurrency"] <= 16 and cfg["model"]
print(cfg)


In [ ]:
# [c02] imports
# [c02]～[c06]、[c08] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils / 網路），tests/ 直接載入。常數對齊 004 / T4 / digest 的 [c02]。
import json
import re
import time
from collections import Counter, OrderedDict, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SLUG = "pm_pack"
PROMPT_VERSION = "0.1"
APPS = ("MNT", "NB", "TV")
BRAND_ALL = "ALL"
BRAND_UNKNOWN = "UNKNOWN"
FACET_ORDER = ("market_outlook", "product_tech", "price", "other")
FACET_LABEL = {"market_outlook": "市場展望", "product_tech": "產品技術", "price": "價格", "other": "其他"}
RELATION_ORDER = ("conflict", "supplement", "corroborate", "meeting_only", "sb_only")
RELATION_ZH = {"conflict": "衝突", "supplement": "補充", "corroborate": "一致", "meeting_only": "只有會議", "sb_only": "只有填報"}
RELATIONS_FOCUS = ("conflict", "supplement")        # A 節預設展開的關係；其餘折疊
RELATIONS_LLM_SET = {"corroborate", "supplement", "conflict"}
VERDICT_ZH = {"agree": "同意", "fix_relation": "改判", "drop": "剔除"}

# 主題與陳述類型中文（順序 = 商務在前、工程在後；同 digest [c02]）
TOPIC_ZH = {
    "demand": "需求 / 預測", "allocation": "Allocation / 限簽", "price": "價格 / 報價", "cost": "成本",
    "product_plan": "產品規劃 / 開案", "spec": "規格", "schedule": "時程", "supply_chain": "供應鏈 / 料況",
    "capacity": "產能 / 廠區", "quality": "品質", "engineering": "工程 / 驗證", "commercial_relation": "商務關係 / 對手",
    "terminology": "術語 / 口徑", "admin": "行政",
}
STATEMENT_ZH = {"fact": "事實", "plan": "規劃", "request": "要求", "decision": "決定", "issue": "問題",
                "opinion": "看法", "definition": "定義"}
HIGHLIGHT_TYPES = ("request", "decision", "issue")   # B 節「代表句」規則：要求 / 決定 / 問題 優先
BUSINESS_TOPICS = ("demand", "allocation", "price", "cost", "product_plan", "commercial_relation", "capacity")  # 其次：商務主題
MAX_HIGHLIGHTS = 5
MAX_LLM_ITEMS = 60                                   # 送 LLM 的條目上限（代表句優先，其餘依 confidence）
SIDE_ZH = {"auo": "AUO", "customer": "客戶", "third_party": "第三方", "unknown": "—"}
# 參與者表只取這四欄（不碰姓名 / email）；summary 裡的 <P07> 顯示成「客戶方（acer）」「AUO 方」（D33）
PART_COLS = ["pk_seq", "token", "side", "company"]
TOKEN_RE = re.compile(r"<([PG]\d{2,})>")
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")
SIDE_LABEL = {"auo": "AUO 方", "customer": "客戶方", "third_party": "第三方", "unknown": "與會者"}
CUSTOMER_NONE = "（客戶不明）"
PRICE_PER_MTOK = {"input": 2.0, "output": 10.0, "cache_write": 2.5, "cache_read": 0.2}  # 只印 log

# 頂端總覽卡固定問 PM 的三題
PM_QUESTIONS = (
    "A 節「衝突」「補充」的格：關係判得對嗎？判錯的請給品牌 × 面向與正確關係。",
    "B 節各客戶的摘要：哪些客戶的摘要有用、哪些是雜訊或漏了重點？",
    "版面：下個月想多看什麼、少看什麼？",
)

# A 節：從 v_meeting_sb_reconcile_q 讀的欄
CELL_COLS = ["app_name", "brand", "facet", "brief_month", "window_start", "window_end", "sb_text", "sb_source",
             "meeting_summary", "n_items", "relation", "relation_basis", "rationale", "number_caveat",
             "cross_customer_note", "sensitivity_max", "reviewer", "review_verdict", "review_relation", "review_note",
             "prompt_version", "effective_relation", "review_pending"]
# B 節：從 v_meeting_items_q 讀的欄（全部非 S3；summary / quote 已 token 化）
ITEM_COLS = ["item_id", "item_key", "pk_seq", "segment_no", "sub_no", "subject_clean", "statement_type", "speaker_side",
             "speaker_company", "customer", "app", "model_nos_str", "topic_tag_primary", "summary", "quote", "event_date",
             "event_month", "sensitivity", "confidence"]
EXTRACTABLE_TYPES = ("inline_minutes", "thread_reply")


In [ ]:
# [c03] pure_render_sb
# A 節：對照格 dict（v_meeting_sb_reconcile_q 一格一列）→ markdown。不重算 relation，只呈現 effective_relation（PM 改判後的值）。
_WS_RE = re.compile(r"\s+")


def _c(v, max_text: int | None = None) -> str:
    """表格安全文字：None → ''；換行 / 連續空白 → 一個空格；`|` → ／；超過 max_text 截斷加 …。"""
    s = "" if v is None else str(v)
    s = _WS_RE.sub(" ", s.replace("|", "／")).strip()
    if max_text and len(s) > max_text:
        s = s[: max_text - 1].rstrip() + "…"
    return s


def _d(v) -> str:
    return f"{v.month:02d}-{v.day:02d}" if isinstance(v, date) else ("" if v is None else str(v))


def _rank(seq, v, default=99) -> int:
    return seq.index(v) if v in seq else default


def cell_sort_key(c: dict) -> tuple:
    """app 固定順序 → 品牌（字母序；UNKNOWN 倒數第二、ALL 最後）→ 面向固定順序。"""
    brand = c.get("brand") or ""
    brand_rank = 2 if brand == BRAND_ALL else (1 if brand == BRAND_UNKNOWN else 0)
    return (_rank(APPS, c.get("app_name")), brand_rank, brand.lower(), _rank(FACET_ORDER, c.get("facet")))


def relation_of(c: dict) -> str:
    return c.get("effective_relation") or c.get("relation") or ""


def review_label(c: dict) -> str:
    """審核欄：待審 / 同意 / 改判 → <新關係> / 剔除；sb_only 不審顯示 —。"""
    if (c.get("relation") or "") == "sb_only":
        return "—"
    v = c.get("review_verdict")
    if not v:
        return "待審"
    if v == "fix_relation":
        return f"改判 → {RELATION_ZH.get(c.get('review_relation') or '', c.get('review_relation') or '?')}"
    return VERDICT_ZH.get(v, v)


def summarize_sb(cells: list[dict]) -> dict:
    """{month: {"n": 格數, "brand_cells", "app_cells", "relations": Counter(effective), "pending", "reviewed",
    "apps": {app: {"n", "relations": Counter, "window": (start, end), "brands": set}}}}"""
    out: dict = {}
    for c in cells:
        m = out.setdefault(c.get("brief_month") or "?", {"n": 0, "brand_cells": 0, "app_cells": 0, "relations": Counter(),
                                                            "pending": 0, "reviewed": 0, "apps": {}})
        rel, is_app = relation_of(c), (c.get("brand") == BRAND_ALL)
        m["n"] += 1
        m["app_cells" if is_app else "brand_cells"] += 1
        m["relations"][rel] += 1
        if c.get("relation") != "sb_only":
            m["pending" if not c.get("review_verdict") else "reviewed"] += 1
        a = m["apps"].setdefault(c.get("app_name") or "?", {"n": 0, "relations": Counter(), "window": (None, None), "brands": set(), "pending": 0})
        a["n"] += 1
        a["relations"][rel] += 1
        if c.get("relation") != "sb_only" and not c.get("review_verdict"):
            a["pending"] += 1
        if not is_app and c.get("brand") != BRAND_UNKNOWN:
            a["brands"].add(c.get("brand"))
        ws, we = c.get("window_start"), c.get("window_end")
        if ws and we:
            cur = a["window"]
            a["window"] = (min(ws, cur[0]) if cur[0] else ws, max(we, cur[1]) if cur[1] else we)
    return out


def _rel_line(counter: Counter) -> str:
    return " / ".join(f"{RELATION_ZH.get(r, r)} {counter[r]}" for r in RELATION_ORDER if counter.get(r))


def render_sb_md(cells: list[dict], months: list[str], *, max_text: int = 300) -> str:
    """每月一節；節內每 app 一張表（品牌 × 面向一列）；衝突格的理由另列在表下。"""
    summary = summarize_sb(cells)
    out: list[str] = ["# A. 會議紀錄 × Sales Brief 對照結果", "",
                      "關係定義：**衝突** = 會議端與填報方向不同；**補充** = 填報沒提、會議有新資訊；**一致** = 會議印證填報；"
                      "**只有會議** = 該格填報空白或該期沒這個品牌；**只有填報** = 會議沒相關條目。"
                      "「會議紀錄說了什麼」是 LLM 依該格條目寫的摘要（內部版，可能含 S2 數字），發言方以陣營顯示。", ""]
    for month in months:
        m = summary.get(month)
        out.append(f"## {month}")
        if not m:
            out += ["", f"這一期沒有對照格：T3 → T4 尚未跑 {month}，或該期沒有相關條目也沒有填報。", ""]
            continue
        out += ["", f"對照格 {m['n']}（品牌格 {m['brand_cells']} ＋ app 層 {m['app_cells']}）：{_rel_line(m['relations'])}；"
                f"PM 已審 {m['reviewed']} / 待審 {m['pending']}（只有填報的格不審）。", ""]
        month_cells = sorted((c for c in cells if (c.get("brief_month") or "?") == month), key=cell_sort_key)
        for app in [a for a in APPS if a in m["apps"]] + sorted(a for a in m["apps"] if a not in APPS):
            a = m["apps"][app]
            ws, we = a["window"]
            win = f"；會議窗口 {_d(ws)}～{_d(we)}" if ws and we else ""
            out += [f"### {app}（填報品牌 {len(a['brands'])}；{_rel_line(a['relations'])}{win}）", "",
                    "| 品牌 | 面向 | 關係 | Sales Brief 填報 | 會議紀錄說了什麼 | 條目 | 審核 |", "|---|---|---|---|---|---|---|"]
            conflicts: list[dict] = []
            for c in (x for x in month_cells if x.get("app_name") == app):
                rel = relation_of(c)
                brand = "app 層（跨品牌）" if c.get("brand") == BRAND_ALL else ("（品牌不在填報名單）" if c.get("brand") == BRAND_UNKNOWN else _c(c.get("brand")))
                out.append(f"| {brand} | {FACET_LABEL.get(c.get('facet'), _c(c.get('facet')))} | {RELATION_ZH.get(rel, rel)} | "
                           f"{_c(c.get('sb_text'), max_text) or '—'} | {_c(c.get('meeting_summary'), max_text) or '—'} | "
                           f"{c.get('n_items') or 0} | {review_label(c)} |")
                if rel == "conflict":
                    conflicts.append(c)
            out.append("")
            if conflicts:
                out.append("衝突理由：")
                for c in conflicts:
                    who = "app 層" if c.get("brand") == BRAND_ALL else _c(c.get("brand"))
                    extra = f"（數字口徑：{_c(c.get('number_caveat'), max_text)}）" if c.get("number_caveat") else ""
                    out.append(f"- {who} × {FACET_LABEL.get(c.get('facet'), _c(c.get('facet')))}：{_c(c.get('rationale'), max_text) or '（無理由）'}{extra}")
                out.append("")
            notes = [c for c in month_cells if c.get("app_name") == app and c.get("cross_customer_note")]
            if notes:
                out.append("跨客戶訊號：")
                out += [f"- {FACET_LABEL.get(c.get('facet'), _c(c.get('facet')))}：{_c(c.get('cross_customer_note'), max_text)}" for c in notes]
                out.append("")
    return "\n".join(out).rstrip() + "\n"


In [ ]:
# [c04] pure_render_monthly
# B 節：條目 dict（v_meeting_items_q 一條一列，帶 dup_key）→ 客戶 × 應用分組 → 每組「輪廓 ＋ 代表句（規則）＋ LLM 摘要（[c08] 另產）」→ markdown。
# 全列放最後（PM 要查細節才看）。不篩業務相關、不對 Sales Brief；只去同文重複（可關）。
def token_labels(parts: list[dict]) -> dict[tuple, str]:
    """participants 列（pk_seq / token / side / company）→ {(pk_seq, token): 顯示標籤}：客戶方（acer）/ AUO 方 / 第三方（inx）/ 與會者；Gnn → 群組（公司）。"""
    out: dict[tuple, str] = {}
    for p in parts:
        key = (p.get("pk_seq"), p.get("token"))
        if key in out and out[key] != SIDE_LABEL["unknown"]:
            continue
        side = SIDE_LABEL.get(p.get("side") or "unknown", SIDE_LABEL["unknown"])
        if str(p.get("token") or "").startswith("G"):
            side = "群組"
        company = p.get("company")
        out[key] = f"{side}（{company}）" if company and side not in ("AUO 方", "與會者") else side
    return out


def resolve_tokens(text, pk_seq, labels: dict | None) -> str:
    """<P07> / <G01> → 陣營標籤；labels 沒有的 token → 與會者。labels None 就原樣（只去 None）。"""
    if not text or labels is None:
        return "" if text is None else str(text)
    return TOKEN_RE.sub(lambda m: labels.get((pk_seq, m.group(1)), SIDE_LABEL["unknown"]), text)


def _speaker(it: dict, labels: dict | None) -> str:
    side = SIDE_ZH.get(it.get("speaker_side") or "unknown", it.get("speaker_side") or "—")
    company = it.get("speaker_company")
    return f"{side}（{company}）" if company and side not in ("AUO", "—") else side


def _basis_zh(basis: str) -> str:
    return "會議日期" if basis == "meeting" else "收信日期"


def group_items(items: list[dict], *, include_dups: bool = False) -> "OrderedDict[tuple, list[dict]]":
    """(customer, app) → 條目（日期 → 主題順序 → item_id）；群組依條目數降冪，客戶不明排最後。"""
    rows = items if include_dups else [it for it in items if not it.get("dup_key")]
    groups: dict[tuple, list[dict]] = defaultdict(list)
    for it in rows:
        groups[(it.get("customer"), it.get("app"))].append(it)
    for g in groups.values():
        g.sort(key=lambda it: (it.get("event_date") or date.max, _rank(tuple(TOPIC_ZH), it.get("topic_tag_primary")), str(it.get("item_id"))))
    ordered = sorted(groups.items(), key=lambda kv: (kv[0][0] is None, -len(kv[1]), str(kv[0][0]), _rank(APPS, kv[0][1], 98)))
    return OrderedDict(ordered)


def highlight_rank(it: dict) -> tuple:
    """代表句排序：要求 / 決定 / 問題 → 商務主題 → S2 → confidence 高 → 日期新。數字越小越前。"""
    st, tp = it.get("statement_type"), it.get("topic_tag_primary")
    return (0 if st in HIGHLIGHT_TYPES else 1, 0 if tp in BUSINESS_TOPICS else 1, 0 if it.get("sensitivity") == "S2" else 1,
            -(it.get("confidence") or 0), -((it.get("event_date") or date.min).toordinal()))


def pick_highlights(rows: list[dict], n: int = MAX_HIGHLIGHTS) -> list[dict]:
    """規則挑代表句（不用 LLM）：highlight_rank 前 n 條；同一摘要文字只留一條。"""
    seen, out = set(), []
    for it in sorted(rows, key=highlight_rank):
        key = _c(it.get("summary")).lower()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(it)
        if len(out) >= n:
            break
    return out


def group_profile(rows: list[dict]) -> dict:
    """一組的輪廓：條目數、信數、日期範圍、前三主題、陳述類型分布、S2 數、客戶方 / AUO 方 發言數。"""
    topics = Counter(it.get("topic_tag_primary") or "" for it in rows)
    types = Counter(it.get("statement_type") or "" for it in rows)
    dates = [it["event_date"] for it in rows if it.get("event_date")]
    return {"n": len(rows), "mails": len({it.get("pk_seq") for it in rows}),
            "date_min": min(dates) if dates else None, "date_max": max(dates) if dates else None,
            "top_topics": [(t, c) for t, c in topics.most_common(3) if t],
            "types": {k: v for k, v in types.items() if k},
            "s2": sum(1 for it in rows if it.get("sensitivity") == "S2"),
            "highlight_n": sum(1 for it in rows if it.get("statement_type") in HIGHLIGHT_TYPES),
            "customer_side": sum(1 for it in rows if it.get("speaker_side") == "customer"),
            "auo_side": sum(1 for it in rows if it.get("speaker_side") == "auo")}


def profile_line(p: dict) -> str:
    topics = "、".join(f"{TOPIC_ZH.get(t, t)} {c}" for t, c in p["top_topics"]) or "—"
    span = f"{_d(p['date_min'])}～{_d(p['date_max'])}" if p.get("date_min") else "日期不明"
    return (f"{p['n']} 條 / {p['mails']} 封（{span}）；主題：{topics}；要求 / 決定 / 問題 {p['highlight_n']} 條；"
            f"客戶方 {p['customer_side']} / AUO 方 {p['auo_side']}；S2 {p['s2']}")


def build_overview(months: list[str], sb_summary: dict, b_profiles: dict) -> list[dict]:
    """總覽列：每月每 app 一列。sb_summary = summarize_sb(cells)；b_profiles = {month: {(customer, app): profile}}。"""
    out = []
    for month in months:
        apps = set(sb_summary.get(month, {}).get("apps", {})) | {a for (_, a) in b_profiles.get(month, {}) if a}
        for app in [a for a in APPS if a in apps] + sorted(a for a in apps if a not in APPS):
            a = sb_summary.get(month, {}).get("apps", {}).get(app, {})
            rel = a.get("relations") or Counter()
            profs = {c: p for (c, ap), p in b_profiles.get(month, {}).items() if ap == app}
            out.append({"month": month, "app": app,
                        "items": sum(p["n"] for p in profs.values()), "customers": sum(1 for c in profs if c),
                        "cust_null": sum(p["n"] for c, p in profs.items() if not c),
                        "cells": a.get("n", 0), "conflict": rel.get("conflict", 0), "supplement": rel.get("supplement", 0),
                        "corroborate": rel.get("corroborate", 0), "meeting_only": rel.get("meeting_only", 0), "sb_only": rel.get("sb_only", 0),
                        "pending": a.get("pending", 0)})
    return out


def render_overview_md(rows: list[dict]) -> str:
    out = ["# 總覽", "", "| 月 | app | 會議條目 | 客戶數 | 客戶不明 | 對照格 | 衝突 | 補充 | 一致 | 只有會議 | 只有填報 |", "|---|---|---|---|---|---|---|---|---|---|---|"]
    out += [f"| {r['month']} | {r['app']} | {r['items']} | {r['customers']} | {r['cust_null']} | {r['cells']} | {r['conflict']} | {r['supplement']} | "
            f"{r['corroborate']} | {r['meeting_only']} | {r['sb_only']} |" for r in rows]
    out += ["", "請 PM 回答：", *[f"{i}. {q}" for i, q in enumerate(PM_QUESTIONS, 1)], ""]
    return "\n".join(out)


def render_funnel_md(month: str, f: dict, *, basis: str) -> list[str]:
    g = lambda k: int(f.get(k) or 0)  # noqa: E731
    lines = [f"### 資料量（信件層按收信月 {month}；條目層按{_basis_zh(basis)} {month}）", "",
             "| 層 | 數量 | 說明 |", "|---|---|---|",
             f"| 收進的信 | {g('mails')} | 其中 ETL 截斷無正文 {g('mails_no_body')}、同信重複 {g('mails_dup')}、隔離（解析不了）{g('mails_skipped')} |",
             f"| 清洗段 | {g('segments')} | 可萃取 {g('extractable')} 段 |",
             f"| 有效條目 | {g('items_valid')} | 來自 {g('items_mails')} 封信；客戶不明 {g('items_cust_null')}、S2 {g('items_s2')} |",
             f"| 正本條目 | {g('items_canonical')} | 去掉逐週重貼的 {g('items_valid') - g('items_canonical')} 條 |"]
    if basis == "meeting" and g("items_no_date"):
        lines.append(f"| 沒有會議日期 | {g('items_no_date')} | 整體無法歸月的正本條目，任何月份都不會出現 |")
    return lines + [""]


def _hl_line(it: dict, labels: dict | None) -> str:
    s2 = "〔S2〕" if it.get("sensitivity") == "S2" else ""
    return (f"- {_d(it.get('event_date')) or '—'}｜{STATEMENT_ZH.get(it.get('statement_type'), _c(it.get('statement_type')))}｜"
            f"{TOPIC_ZH.get(it.get('topic_tag_primary') or '', '—')}｜{_speaker(it, labels)}：{s2}{_c(resolve_tokens(it.get('summary'), it.get('pk_seq'), labels))}")


def render_group_cards_md(month: str, groups: "OrderedDict", labels: dict | None, summaries: dict) -> list[str]:
    """每組：輪廓一行 → LLM 摘要（有的話）→ 值得注意 → 代表句。summaries = {(customer, app): {"summary", "watch", "error"}}。"""
    lines = [f"### 客戶 × 應用摘要（{len(groups)} 組）", ""]
    for (customer, app), rows in groups.items():
        p = group_profile(rows)
        lines += [f"#### {customer or CUSTOMER_NONE} × {app or '—'}", "", profile_line(p), ""]
        s = summaries.get((customer, app)) or {}
        if s.get("summary"):
            lines += [f"**摘要（LLM）**：{s['summary']}", ""]
            if s.get("watch"):
                lines += ["值得注意：", *[f"- {w}" for w in s["watch"]], ""]
        elif s.get("error"):
            lines += [f"（LLM 摘要失敗：{s['error']}）", ""]
        lines += ["代表句（規則：要求 / 決定 / 問題 優先）：", *[_hl_line(it, labels) for it in pick_highlights(rows)], ""]
    return lines


def render_items_md(month: str, items: list[dict], labels: dict | None, *, include_dups: bool = False,
                    include_quote: bool = False, max_per_group: int = 0) -> list[str]:
    groups = group_items(items, include_dups=include_dups)
    total = sum(len(g) for g in groups.values())
    lines = [f"### 條目全列（{total} 條，{len(groups)} 個客戶 × 應用；{'含' if include_dups else '不含'}同文重複）", ""]
    head = "| 日期 | 類型 | 主題 | 發言方 | 摘要 | S | 來源 |" + (" 原文 |" if include_quote else "")
    sep = "|---|---|---|---|---|---|---|" + ("---|" if include_quote else "")
    for (customer, app), rows in groups.items():
        lines += [f"#### {customer or CUSTOMER_NONE} × {app or '—'}（{len(rows)} 條）", "", head, sep]
        shown = rows[:max_per_group] if max_per_group else rows
        for it in shown:
            src = f"{it.get('pk_seq')}-{it.get('segment_no')}" + (f"-{it['sub_no']}" if it.get("sub_no") else "")
            subj = _c(it.get("subject_clean"), 40)
            row = (f"| {_d(it.get('event_date')) or '—'} | {STATEMENT_ZH.get(it.get('statement_type'), _c(it.get('statement_type')))} | "
                   f"{TOPIC_ZH.get(it.get('topic_tag_primary') or '', _c(it.get('topic_tag_primary')) or '—')} | {_speaker(it, labels)} | "
                   f"{_c(resolve_tokens(it.get('summary'), it.get('pk_seq'), labels))} | {_c(it.get('sensitivity')) or 'S0'} | "
                   f"{src}{'（' + subj + '）' if subj else ''} |")
            if include_quote:
                row += f" {_c(resolve_tokens(it.get('quote'), it.get('pk_seq'), labels), 300)} |"
            lines.append(row)
        if len(rows) > len(shown):
            lines.append(f"| … | | | | 另 {len(rows) - len(shown)} 條（max_per_group = {max_per_group}） | | |" + (" |" if include_quote else ""))
        lines.append("")
    return lines


def render_monthly_md(month: str, funnel: dict, items: list[dict], labels: dict | None, *, basis: str = "meeting",
                      include_dups: bool = False, include_quote: bool = False, max_per_group: int = 0,
                      summaries: dict | None = None) -> str:
    groups = group_items(items, include_dups=include_dups)
    out = [f"## {month} 會議紀錄彙整（按{_basis_zh(basis)}）", ""]
    out += render_funnel_md(month, funnel, basis=basis)
    out += render_group_cards_md(month, groups, labels, summaries or {})
    out += render_items_md(month, items, labels, include_dups=include_dups, include_quote=include_quote, max_per_group=max_per_group)
    out += ["資料缺口：約三成信件 ETL 截斷無正文；附件（Excel / PPT）不讀；客戶欄約四成空值（多為內部會議與供應鏈段）；"
            "人名以陣營顯示、不出現姓名；同一段紀要逐週重貼已合併為正本。", ""]
    return "\n".join(out).rstrip() + "\n"


In [ ]:
# [c05] pure_spark
# DataFrame -> DataFrame：先過濾再 select；每月幾百到幾千列，collect 到 driver 做 md。
# 兩種月：mail_month = 收信日（v_meeting_mail_index.mail_date）；meeting_month = 會議日期（event_date）。
def _ym(col: str):
    return F.date_format(F.col(col), "yyyy-MM")


def _n(cond):
    return F.sum(F.when(cond, 1).otherwise(0))  # Spark 4 ANSI：不能 sum(boolean)


def select_sb_cells(df_rec_q: DataFrame, months: list[str]) -> DataFrame:
    """v_meeting_sb_reconcile_q → 指定期別的格（CELL_COLS）。"""
    return df_rec_q.where(F.col("brief_month").isin(*months)).select(*CELL_COLS)


def attach_months(df_items_q: DataFrame, df_mail_idx: DataFrame, df_lens: DataFrame) -> DataFrame:
    """v_meeting_items_q ⋈ v_meeting_mail_index（pk_seq → mail_date）⋈ 鏡頭表 active（item_key → dup_key）
    → ITEM_COLS ＋ mail_month ＋ meeting_month ＋ dup_key。沒進鏡頭表的條目 dup_key = NULL（當正本）。"""
    mail = df_mail_idx.select("pk_seq", _ym("mail_date").alias("mail_month"))
    lens = df_lens.where(F.col("active")).select("item_key", "dup_key").dropDuplicates(["item_key"])
    return (df_items_q.select(*ITEM_COLS)
            .join(mail, "pk_seq", "left")
            .join(lens, "item_key", "left")
            .withColumn("meeting_month", _ym("event_date")))


def _month_col(basis: str) -> str:
    return "meeting_month" if basis == "meeting" else "mail_month"


def mail_funnel(df_mail_idx: DataFrame, months: list[str]) -> DataFrame:
    """信件層（永遠按收信月）：mails / mails_no_body / mails_dup。"""
    return (df_mail_idx.withColumn("month", _ym("mail_date")).where(F.col("month").isin(*months))
            .groupBy("month").agg(F.count("*").alias("mails"),
                                  _n(F.col("body_source_field") == "none").alias("mails_no_body"),
                                  _n(F.col("dup_of_pk").isNotNull()).alias("mails_dup")))


def skipped_funnel(df_skipped: DataFrame, months: list[str]) -> DataFrame:
    """隔離表（002 D34）：mails_skipped。"""
    return (df_skipped.withColumn("month", _ym("mail_date")).where(F.col("month").isin(*months))
            .groupBy("month").agg(F.count("*").alias("mails_skipped")))


def clean_funnel(df_clean: DataFrame, months: list[str]) -> DataFrame:
    """清洗段（按收信月）：segments / extractable。"""
    return (df_clean.withColumn("month", _ym("mail_date")).where(F.col("month").isin(*months))
            .groupBy("month").agg(F.count("*").alias("segments"),
                                  _n(F.col("content_type").isin(*EXTRACTABLE_TYPES) & F.col("dup_of").isNull()).alias("extractable")))


def items_funnel(df_items_m: DataFrame, months: list[str], *, basis: str = "meeting") -> DataFrame:
    """條目層（按 basis 的月）：items_valid / items_canonical / items_cust_null / items_s2 / items_mails。"""
    mc = _month_col(basis)
    return (df_items_m.where(F.col(mc).isin(*months)).groupBy(F.col(mc).alias("month"))
            .agg(F.count("*").alias("items_valid"),
                 _n(F.col("dup_key").isNull()).alias("items_canonical"),
                 _n(F.col("dup_key").isNull() & F.col("customer").isNull()).alias("items_cust_null"),
                 _n(F.col("dup_key").isNull() & (F.col("sensitivity") == "S2")).alias("items_s2"),
                 F.countDistinct("pk_seq").alias("items_mails")))


def items_no_date(df_items_m: DataFrame) -> DataFrame:
    """沒有會議日期的正本條目數（整體，不分月；basis = meeting 時提醒）。"""
    return df_items_m.where(F.col("event_date").isNull() & F.col("dup_key").isNull()).agg(F.count("*").alias("items_no_date"))


def cust_app_counts(df_items_m: DataFrame, month: str, *, basis: str = "meeting", include_dups: bool = False) -> DataFrame:
    """該月條目按客戶 × 應用計數（預設只算正本），降冪。"""
    df = df_items_m.where(F.col(_month_col(basis)) == F.lit(month))
    if not include_dups:
        df = df.where(F.col("dup_key").isNull())
    return df.groupBy("customer", "app").agg(F.count("*").alias("n")).orderBy(F.desc("n"), "customer", "app")


def select_month_items(df_items_m: DataFrame, month: str, *, basis: str = "meeting", include_dups: bool = False) -> DataFrame:
    """該月要列的條目（ITEM_COLS ＋ dup_key ＋ 兩個月欄）；預設只留正本。"""
    df = df_items_m.where(F.col(_month_col(basis)) == F.lit(month))
    if not include_dups:
        df = df.where(F.col("dup_key").isNull())
    return df.select(*ITEM_COLS, "dup_key", "mail_month", "meeting_month")


In [ ]:
# [c06] pure_render_html
# HTML 版（給 PM 用瀏覽器開，純閱讀、無 JS、無外部資源）：版面沿 Sales Brief 快報（漸層 header、section-card、藍表頭、tag）。
# 三層：頂端總覽卡（每月每 app 一行數字 ＋ 三個問題）→ A 節只展開衝突 / 補充，其餘折疊 → B 節每組一張摘要卡（輪廓、LLM 摘要、代表句），全列折疊在最後。
import html as _html

CSS = """
*{box-sizing:border-box;margin:0;padding:0;}
body{font-family:'Segoe UI','Noto Sans TC','Microsoft JhengHei',Arial,sans-serif;background:#F3F4F6;color:#1A1A2E;font-size:14px;}
.container{max-width:1100px;margin:0 auto;padding:16px;}
.header{background:linear-gradient(135deg,#1F4E79,#2E75B6);color:#fff;border-radius:12px;padding:26px 30px;margin-bottom:20px;}
.header h1{font-size:24px;font-weight:700;margin-bottom:8px;}
.header-meta{display:flex;flex-wrap:wrap;gap:12px;font-size:13px;opacity:0.92;}
.header-meta span{background:rgba(255,255,255,0.15);padding:4px 12px;border-radius:20px;}
.nav{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:18px;font-size:13px;}
.nav a{background:#fff;border:1px solid #C8D8E8;border-radius:20px;padding:4px 12px;color:#1F4E79;text-decoration:none;}
.section-card{background:#fff;border-radius:12px;border:1px solid #C8D8E8;padding:22px 24px;margin-bottom:20px;}
.section-title{display:flex;align-items:center;gap:12px;margin-bottom:12px;}
.section-badge{width:28px;height:28px;border-radius:50%;background:#1F4E79;color:#fff;font-size:13px;font-weight:700;display:flex;align-items:center;justify-content:center;flex-shrink:0;}
.section-title h2{font-size:17px;font-weight:700;}
.sub-title{font-size:14px;font-weight:700;color:#1F4E79;margin:18px 0 10px;}
.insight-box{background:#F5F7FA;border-left:4px solid #1F4E79;padding:12px 16px;border-radius:0 6px 6px 0;margin-top:12px;font-size:13px;line-height:1.8;}
.ask-box{background:#FFF8E1;border-left:4px solid #BF8F00;padding:12px 16px;border-radius:0 6px 6px 0;margin-top:12px;font-size:13px;line-height:1.8;}
.warn-box{background:#FFF8E1;border:1px solid #FFD54F;border-radius:8px;padding:12px 18px;margin-bottom:18px;font-size:13px;color:#5D4037;line-height:1.7;}
table{width:100%;border-collapse:collapse;font-size:13px;}
th{background:#1F4E79;color:#fff;padding:8px 9px;text-align:center;font-weight:600;white-space:nowrap;}
td{padding:7px 9px;border-bottom:1px solid #DCE6F0;vertical-align:top;line-height:1.6;}
tr:nth-child(even) td{background:#EFF5FB;}
tr:nth-child(odd) td{background:#fff;}
.td-num{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap;}
.td-c{text-align:center;white-space:nowrap;}
.tbl-scroll{overflow-x:auto;}
.tag-pos{display:inline-block;background:#E8F5E9;color:#2E7D32;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neg{display:inline-block;background:#FDECEC;color:#C00000;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neu{display:inline-block;background:#FFF8E1;color:#946C00;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-grey{display:inline-block;background:#ECEFF1;color:#546E7A;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-s2{display:inline-block;background:#FDECEC;color:#C00000;border-radius:4px;padding:0 5px;font-size:11px;}
.note{font-size:11px;color:#8A8A96;margin-top:8px;line-height:1.6;}
.footer{text-align:center;font-size:12px;color:#9E9E9E;padding:14px 0 24px;line-height:1.7;}
.brand-cell{width:110px;font-weight:700;}
.facet-cell{width:84px;background:#EDF2F7 !important;color:#1F4E79;font-weight:700;}
details{margin:10px 0;border:1px solid #DCE6F0;border-radius:8px;padding:6px 12px;background:#FCFCFB;}
summary{cursor:pointer;font-weight:700;color:#1F4E79;padding:4px 0;}
.gcard{border:1px solid #DCE6F0;border-radius:10px;padding:14px 16px;margin:12px 0;background:#fff;}
.gcard h4{font-size:15px;color:#1F4E79;margin-bottom:4px;}
.gcard .prof{font-size:12px;color:#5A5A6E;margin-bottom:8px;}
.gcard .sum{font-size:13px;line-height:1.8;background:#F5F7FA;border-left:4px solid #1F4E79;padding:10px 14px;border-radius:0 6px 6px 0;}
.gcard ul{margin:6px 0 0 18px;font-size:13px;line-height:1.7;}
.gcard .hl li{color:#333;}
.muted{color:#8A8A96;}
"""

REL_TAG = {"conflict": "tag-neg", "supplement": "tag-neu", "corroborate": "tag-pos", "meeting_only": "tag-grey", "sb_only": "tag-grey"}


def esc(v, max_text: int | None = None) -> str:
    return _html.escape(_c(v, max_text), quote=True)


def _brand_html(c: dict) -> str:
    b = c.get("brand")
    return "app 層（跨品牌）" if b == BRAND_ALL else ("（品牌不在填報名單）" if b == BRAND_UNKNOWN else esc(b))


def _rel_html(rel: str) -> str:
    return f'<span class="{REL_TAG.get(rel, "tag-grey")}">{esc(RELATION_ZH.get(rel, rel))}</span>'


def _review_html(c: dict) -> str:
    """審核欄（唯讀）：待審 / 已審結果 ＋ 備註；sb_only 不審。"""
    if (c.get("relation") or "") == "sb_only":
        return "—"
    if not c.get("review_verdict"):
        return '<span class="tag-grey">待審</span>'
    return f"{esc(review_label(c))}" + (f"<br><span class='note'>{esc(c.get('review_note'))}</span>" if c.get("review_note") else "")


def _ul(items: list[str]) -> str:
    return "<ul>" + "".join(f"<li>{x}</li>" for x in items) + "</ul>"


# ---------- 總覽 ----------
def render_overview_html(rows: list[dict], *, badge: int = 1, has_a: bool = True, has_b: bool = True) -> str:
    head = f'<div class="section-title"><div class="section-badge">{badge}</div><h2 id="overview">總覽：這一包有什麼、請你看哪裡</h2></div>'
    th = "<th>月</th><th>app</th>" + ("<th>會議條目</th><th>客戶數</th><th>客戶不明</th>" if has_b else "") + \
         ("<th>對照格</th><th>衝突</th><th>補充</th><th>一致</th><th>只有會議</th><th>只有填報</th><th>待審</th>" if has_a else "")
    trs = []
    for r in rows:
        tds = f'<td class="td-c">{esc(r["month"])}</td><td class="td-c"><b>{esc(r["app"])}</b></td>'
        if has_b:
            tds += f'<td class="td-num">{r["items"]}</td><td class="td-num">{r["customers"]}</td><td class="td-num">{r["cust_null"]}</td>'
        if has_a:
            tds += (f'<td class="td-num">{r["cells"]}</td><td class="td-num"><b style="color:#C00000">{r["conflict"]}</b></td>'
                    f'<td class="td-num"><b style="color:#946C00">{r["supplement"]}</b></td><td class="td-num">{r["corroborate"]}</td>'
                    f'<td class="td-num">{r["meeting_only"]}</td><td class="td-num">{r["sb_only"]}</td><td class="td-num">{r["pending"]}</td>')
        trs.append(f"<tr>{tds}</tr>")
    tbl = f'<div class="tbl-scroll"><table><thead><tr>{th}</tr></thead><tbody>{"".join(trs)}</tbody></table></div>'
    guide = []
    if has_a:
        guide.append("<b>A 節</b>：會議紀錄對上 Sales Brief 填報的結果。預設只展開<span class='tag-neg'>衝突</span>和<span class='tag-neu'>補充</span>（會議講了填報沒寫的）；一致 / 只有會議 / 只有填報折疊，點開才看。")
    if has_b:
        guide.append("<b>B 節</b>：每個客戶 × 應用一張卡：條目輪廓、LLM 寫的 3–5 句摘要、規則挑的代表句（要求 / 決定 / 問題優先）。逐條全列折疊在最後，要查細節才點。")
    ask = f'<div class="ask-box"><b>請 PM 回答三題</b>{_ul([esc(q) for q in PM_QUESTIONS])}</div>'
    return f'<div class="section-card">{head}{tbl}<div class="insight-box">{"<br>".join(guide)}</div>{ask}</div>'


# ---------- A 節 ----------
def _cell_tr(c: dict, max_text: int) -> str:
    rel = relation_of(c)
    reason = f'<br><span class="note">理由：{esc(c.get("rationale"), max_text)}</span>' if c.get("rationale") and rel in RELATIONS_LLM_SET else ""
    return (f'<tr><td class="brand-cell">{_brand_html(c)}</td>'
            f'<td class="facet-cell">{esc(FACET_LABEL.get(c.get("facet"), c.get("facet")))}</td>'
            f'<td class="td-c">{_rel_html(rel)}</td>'
            f'<td>{esc(c.get("sb_text"), max_text) or "—"}</td>'
            f'<td>{esc(c.get("meeting_summary"), max_text) or "—"}{reason}</td>'
            f'<td class="td-num">{c.get("n_items") or 0}</td><td class="td-c">{_review_html(c)}</td></tr>')


_TBL_HEAD = ('<div class="tbl-scroll"><table><thead><tr><th>品牌</th><th>面向</th><th>關係</th><th>Sales Brief 填報</th>'
             '<th>會議紀錄說了什麼</th><th>條目</th><th>審核</th></tr></thead><tbody>')


def render_sb_app_html(month: str, app: str, cells: list[dict], *, max_text: int = 150) -> str:
    cells = sorted(cells, key=cell_sort_key)
    focus = [c for c in cells if relation_of(c) in RELATIONS_FOCUS]
    rest = [c for c in cells if relation_of(c) not in RELATIONS_FOCUS]
    a = summarize_sb(cells)[month]["apps"][app]
    ws, we = a["window"]
    win = f"；會議窗口 {_d(ws)}～{_d(we)}" if ws and we else ""
    out = [f'<div class="sub-title">{esc(app)}｜填報品牌 {len(a["brands"])}；{esc(_rel_line(a["relations"]))}{esc(win)}</div>']
    if focus:
        out += [_TBL_HEAD, *[_cell_tr(c, max_text) for c in focus], "</tbody></table></div>"]
    else:
        out.append('<p class="muted">這個 app 本期沒有衝突或補充的格。</p>')
    notes = [c for c in cells if c.get("cross_customer_note")]
    if notes:
        out.append('<div class="insight-box">&#128204; <b>跨客戶訊號</b>' + _ul([f"<b>{esc(FACET_LABEL.get(c.get('facet'), c.get('facet')))}</b>：{esc(c.get('cross_customer_note'), max_text)}" for c in notes]) + "</div>")
    if rest:
        rc = Counter(relation_of(c) for c in rest)
        label = "、".join(f"{RELATION_ZH.get(r, r)} {rc[r]}" for r in RELATION_ORDER if rc.get(r))
        out.append(f"<details><summary>其餘 {len(rest)} 格（{esc(label)}）</summary>{_TBL_HEAD}{''.join(_cell_tr(c, max_text) for c in rest)}</tbody></table></div></details>")
    return "\n".join(out)


def render_sb_html(cells: list[dict], months: list[str], *, max_text: int = 150, badge_start: int = 1) -> tuple[list[str], int]:
    """每月一張 section-card；回傳 (cards, 下一個 badge 號)。"""
    summary, cards, badge = summarize_sb(cells), [], badge_start
    for month in months:
        m = summary.get(month)
        head = (f'<div class="section-title"><div class="section-badge">{badge}</div>'
                f'<h2 id="a-{month}">{month} 會議紀錄 × Sales Brief 對照</h2></div>')
        if not m:
            cards.append(f'<div class="section-card">{head}<p>這一期沒有對照格：T3 → T4 尚未跑 {month}，或該期沒有相關條目也沒有填報。</p></div>')
            badge += 1
            continue
        intro = (f'<p style="font-size:13px;color:#444">對照格 {m["n"]}（品牌格 {m["brand_cells"]} ＋ app 層 {m["app_cells"]}）：{esc(_rel_line(m["relations"]))}；'
                 f'PM 已審 {m["reviewed"]} / 待審 {m["pending"]}（只有填報的格不審）。</p>')
        month_cells = [c for c in cells if (c.get("brief_month") or "?") == month]
        apps = [a for a in APPS if a in m["apps"]] + sorted(a for a in m["apps"] if a not in APPS)
        body = "\n".join(render_sb_app_html(month, a, [c for c in month_cells if c.get("app_name") == a], max_text=max_text) for a in apps)
        cards.append(f'<div class="section-card">{head}{intro}{body}</div>')
        badge += 1
    return cards, badge


# ---------- B 節 ----------
def _hl_li(it: dict, labels: dict | None) -> str:
    s2 = '<span class="tag-s2">S2</span> ' if it.get("sensitivity") == "S2" else ""
    return (f'<li><span class="muted">{esc(_d(it.get("event_date"))) or "—"}｜{esc(STATEMENT_ZH.get(it.get("statement_type"), it.get("statement_type")))}｜'
            f'{esc(TOPIC_ZH.get(it.get("topic_tag_primary") or "", "—"))}｜{esc(_speaker(it, labels))}：</span>{s2}'
            f'{esc(resolve_tokens(it.get("summary"), it.get("pk_seq"), labels))}</li>')


def render_group_card_html(customer, app, rows: list[dict], labels: dict | None, s: dict | None) -> str:
    p = group_profile(rows)
    s = s or {}
    parts = [f'<div class="gcard"><h4>{esc(customer or CUSTOMER_NONE)} × {esc(app or "—")}</h4><div class="prof">{esc(profile_line(p))}</div>']
    if s.get("summary"):
        parts.append(f'<div class="sum"><b>摘要</b>（LLM）：{esc(s["summary"])}</div>')
        if s.get("watch"):
            parts.append("<b style='font-size:13px'>值得注意</b>" + _ul([esc(w) for w in s["watch"]]))
    elif s.get("error"):
        parts.append(f'<div class="sum muted">LLM 摘要失敗：{esc(s["error"])}；請看代表句。</div>')
    elif s.get("skipped"):
        parts.append(f'<div class="sum muted">{esc(s["skipped"])}</div>')
    parts.append("<b style='font-size:13px'>代表句</b><span class='muted'>（規則：要求 / 決定 / 問題優先）</span>" + '<ul class="hl">' + "".join(_hl_li(it, labels) for it in pick_highlights(rows)) + "</ul></div>")
    return "".join(parts)


def _items_table_html(rows: list[dict], labels: dict | None, *, include_quote: bool, max_per_group: int) -> str:
    qh = "<th>原文</th>" if include_quote else ""
    shown = rows[:max_per_group] if max_per_group else rows
    trs = []
    for it in shown:
        src = f"{it.get('pk_seq')}-{it.get('segment_no')}" + (f"-{it['sub_no']}" if it.get("sub_no") else "")
        s = it.get("sensitivity") or "S0"
        stag = f'<span class="tag-s2">{esc(s)}</span>' if s == "S2" else esc(s)
        trs.append(f'<tr><td class="td-c">{esc(_d(it.get("event_date"))) or "—"}</td>'
                   f'<td class="td-c">{esc(STATEMENT_ZH.get(it.get("statement_type"), it.get("statement_type")))}</td>'
                   f'<td>{esc(TOPIC_ZH.get(it.get("topic_tag_primary") or "", it.get("topic_tag_primary") or "—"))}</td>'
                   f'<td>{esc(_speaker(it, labels))}</td>'
                   f'<td>{esc(resolve_tokens(it.get("summary"), it.get("pk_seq"), labels))}</td>'
                   f'<td class="td-c">{stag}</td>'
                   f'<td class="td-c" title="{esc(it.get("subject_clean"))}">{esc(src)}</td>'
                   + (f'<td>{esc(resolve_tokens(it.get("quote"), it.get("pk_seq"), labels), 300)}</td>' if include_quote else "") + "</tr>")
    if len(rows) > len(shown):
        trs.append(f'<tr><td colspan="{8 if include_quote else 7}" class="td-c">… 另 {len(rows) - len(shown)} 條（max_per_group = {max_per_group}）</td></tr>')
    return (f'<div class="tbl-scroll"><table><thead><tr><th>日期</th><th>類型</th><th>主題</th><th>發言方</th><th>摘要</th><th>S</th><th>來源</th>{qh}</tr></thead>'
            f'<tbody>{"".join(trs)}</tbody></table></div>')


def render_monthly_html(month: str, funnel: dict, items: list[dict], labels: dict | None, *, basis: str = "meeting",
                        include_dups: bool = False, include_quote: bool = False, max_per_group: int = 0, badge: int = 1,
                        summaries: dict | None = None) -> str:
    g = lambda k: int(funnel.get(k) or 0)  # noqa: E731
    summaries = summaries or {}
    groups = group_items(items, include_dups=include_dups)
    head = (f'<div class="section-title"><div class="section-badge">{badge}</div>'
            f'<h2 id="b-{month}">{month} 會議紀錄彙整（按{_basis_zh(basis)}）</h2></div>')
    intro = (f'<p style="font-size:13px;color:#444">收信 {g("mails")} 封（無正文 {g("mails_no_body")}、重複 {g("mails_dup")}、隔離 {g("mails_skipped")}）→ '
             f'有效條目 {g("items_valid")} → 正本 {g("items_canonical")}（客戶不明 {g("items_cust_null")}、S2 {g("items_s2")}）；'
             f'{len(groups)} 個客戶 × 應用。' + (f' 另有 {g("items_no_date")} 條沒有會議日期，不會出現在任何月份。' if basis == "meeting" and g("items_no_date") else "") + "</p>")
    cards = "".join(render_group_card_html(c, a, rows, labels, summaries.get((c, a))) for (c, a), rows in groups.items())
    full = "".join(f"<details><summary>{esc(c or CUSTOMER_NONE)} × {esc(a or '—')}（{len(rows)} 條）</summary>"
                   f"{_items_table_html(rows, labels, include_quote=include_quote, max_per_group=max_per_group)}</details>"
                   for (c, a), rows in groups.items())
    total = sum(len(v) for v in groups.values())
    full_block = f'<details><summary>逐條全列（{total} 條；{"含" if include_dups else "不含"}同文重複）— 要查細節才點</summary>{full}</details>'
    note = ('<div class="note">資料缺口：約三成信件 ETL 截斷無正文；附件（Excel / PPT）不讀；客戶欄約四成空值（多為內部會議與供應鏈段）；'
            '人名以陣營顯示、不出現姓名；同一段紀要逐週重貼已合併為正本；S2 = 報價 / allocation 等敏感條目，本頁為內部版。'
            '摘要由 LLM 依條目撰寫、只能用條目內的資訊；代表句由規則挑選。</div>')
    return f'<div class="section-card">{head}{intro}<div class="sub-title">客戶 × 應用摘要</div>{cards}<div class="sub-title">附：逐條全列</div>{full_block}{note}</div>'


def render_page_html(title: str, meta: list[str], cards: list[str], *, warn: str = "",
                     nav: list[tuple[str, str]] | None = None) -> str:
    """整頁：header → 提示 → 導覽 → cards → footer。純閱讀。"""
    meta_html = "".join(f"<span>{esc(m)}</span>" for m in meta)
    nav_html = ('<div class="nav">' + "".join(f'<a href="#{esc(h)}">{esc(t)}</a>' for h, t in nav) + "</div>") if nav else ""
    warn_html = f'<div class="warn-box">{esc(warn)}</div>' if warn else ""
    return ("<!DOCTYPE html>\n<html lang=\"zh-Hant\"><head><meta charset=\"UTF-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1.0\">"
            f"<title>{esc(title)}</title><style>{CSS}</style></head>"
            '<body><div class="container">'
            f'<div class="header"><h1>{esc(title)}</h1><div class="header-meta">{meta_html}</div></div>'
            f"{warn_html}{nav_html}{''.join(cards)}"
            '<div class="footer">本頁由 MI DataHub meeting_extracted_pm_pack 產生；會議內容為各方陳述、已去識別，不是已驗證數據。</div>'
            "</div></body></html>")


In [ ]:
# [c07] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 T4 / Stage A 同一組 secret / URL。用 requests（DBR 內建），不 %pip install。
# thinking 關閉、不送 temperature、system 開 prompt caching（system prompt 固定，整期共用）。
# 重試：HTTP 408 / 429 / 5xx / 連線錯誤指數退避 3 次；4xx 其他直接回 error 不重試。JSON 失敗的重試在 [c11]。llm_summary = false 時不會建立連線。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry") if (cfg["llm_summary"] and "B" in cfg["sections"]) else None
PM_MAX_TOKENS = 1500      # 一組 3–5 句 ＋ 3 條 watch 的 JSON；截斷視為失敗
HTTP_TIMEOUT_S = 120


def call_llm(messages: list[dict], system_prompt: str, model: str, cache: bool) -> dict:
    """回傳 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": PM_MAX_TOKENS, "system": system,
            "messages": messages, "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code in (408, 429) or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client", "ready" if FOUNDRY_API_KEY else "off（llm_summary = false 或沒有 B 節）", "| model:", cfg["model"], "| cache:", cfg["prompt_cache"])


In [ ]:
# [c08] pure_llm
# B 節客戶 × 應用摘要的 prompt 與後處理（純 Python；呼叫在 [c11]）。一組一次：輸入該組條目（已去識別、發言方以陣營標示），
# 輸出 JSON {"summary": 3–5 句, "watch": [最多 3 條]}。紀律：只能用條目裡的資訊、每句標明誰說的、數字照抄、不出現人名 / token / email。
SYSTEM_PROMPT_PM = f"""你是面板廠 MI 團隊的會議紀錄彙整助理。輸入是某個客戶 × 應用在一個月內、從內部會議紀錄萃取出來的條目清單（已去識別：人名換成「客戶方（公司）」「AUO 方」「第三方（公司）」）。
請為主管寫一段「這個客戶這個月在會議上談了什麼」的摘要。

規則：
1. 只能用條目裡出現的資訊。不得推論原因、不得補充條目沒有的事實、不得下結論說哪一方對。
2. 每一句都要讓讀者知道是「客戶方」還是「AUO 方」說的；條目沒標發言方就寫「會議中提到」。
3. 數字、型號、日期照抄，帶原本的單位；不要換算、不要四捨五入。
4. 優先寫：客戶的要求、雙方的決定、未解的問題、需求 / 價格 / allocation / 產品規劃；工程料號與驗證細節只在它是要求或決定時才寫。
5. 不得出現任何人名、員工編號、email、<P 或 <G 開頭的代號。
6. summary 3 到 5 句、繁體中文、總長 350 字以內；條目太少寫不出摘要就把 summary 寫「本月條目少，見代表句」。
7. watch 是「主管下個月該追的事」，最多 3 條、每條 60 字以內、只能來自條目；沒有就給空陣列。

只輸出一個 JSON 物件：{{"summary": "...", "watch": ["...", "..."]}}。不要加任何說明文字或 markdown 圍欄。
（prompt {PROMPT_VERSION}）"""

MAX_SUMMARY_CHARS = 500
MAX_WATCH = 3
MAX_WATCH_CHARS = 100


def _item_line_for_llm(it: dict, labels: dict | None) -> str:
    s2 = "〔S2〕" if it.get("sensitivity") == "S2" else ""
    speaker = _speaker(it, labels)
    return (f"- {_d(it.get('event_date')) or '日期不明'}｜{STATEMENT_ZH.get(it.get('statement_type'), it.get('statement_type') or '')}｜"
            f"{TOPIC_ZH.get(it.get('topic_tag_primary') or '', '—')}｜{speaker}：{s2}{_c(resolve_tokens(it.get('summary'), it.get('pk_seq'), labels))}")


def select_items_for_llm(rows: list[dict], cap: int = MAX_LLM_ITEMS) -> list[dict]:
    """送 LLM 的條目：代表句排序前 cap 條（要求 / 決定 / 問題 ＋ 商務主題優先），再按日期排回去。"""
    chosen = sorted(rows, key=highlight_rank)[:cap]
    return sorted(chosen, key=lambda it: (it.get("event_date") or date.max, str(it.get("item_id"))))


def build_group_prompt(month: str, customer: str | None, app: str | None, rows: list[dict], labels: dict | None,
                       *, basis: str = "meeting") -> str:
    p = group_profile(rows)
    sent = select_items_for_llm(rows)
    head = [f"客戶：{customer or '（不明）'}｜應用：{app or '—'}｜月份：{month}（按{_basis_zh(basis)}）",
            f"條目 {p['n']} 條（送出 {len(sent)} 條）、來自 {p['mails']} 封信；{profile_line(p)}", "", "條目（日期｜類型｜主題｜發言方：內容；〔S2〕= 敏感數字）："]
    return "\n".join(head + [_item_line_for_llm(it, labels) for it in sent])


def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """回覆文字 → (dict, None) 或 (None, 錯誤訊息)。容忍 markdown 圍欄與前後說明文字（取第一個 { 到最後一個 }）。"""
    if not text or not text.strip():
        return None, "empty response"
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None, "no JSON object found"
    try:
        obj = json.loads(s[i:j + 1])
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e.msg} at pos {e.pos}"
    if not isinstance(obj, dict):
        return None, "top-level must be an object"
    return obj, None


def _clean_text(v, cap: int) -> str | None:
    if v is None or isinstance(v, (list, dict, bool)):
        return None
    s = _c(v)
    s = EMAIL_ANY_RE.sub("<email>", s)
    return s[:cap] if s else None


def postprocess_summary(obj: dict | None, err: str | None) -> dict:
    """→ {"summary", "watch", "error"}。硬規則：含 <Pnn> / <Gnn> token 或 email → 整組作廢（error），避免人名外洩。"""
    if obj is None:
        return {"summary": None, "watch": [], "error": err or "no object"}
    summary = _clean_text(obj.get("summary"), MAX_SUMMARY_CHARS)
    watch_raw = obj.get("watch") if isinstance(obj.get("watch"), list) else []
    watch = [w for w in (_clean_text(x, MAX_WATCH_CHARS) for x in watch_raw if isinstance(x, str)) if w][:MAX_WATCH]
    if not summary:
        return {"summary": None, "watch": [], "error": "empty summary"}
    joined = summary + " ".join(watch)
    if TOKEN_RE.search(joined) or "<email>" in joined:
        return {"summary": None, "watch": [], "error": "token / email leaked"}
    return {"summary": summary, "watch": watch, "error": None}


def retry_user_prompt(perr: str) -> str:
    return f"上一則回覆無法解析為 JSON（{perr}）。請只輸出一個 JSON 物件：{{\"summary\": \"...\", \"watch\": [\"...\"]}}，不要圍欄、不要說明。"


def est_cost_usd(usage: dict) -> float:
    return round(sum(usage.get(k, 0) * PRICE_PER_MTOK[k] for k in PRICE_PER_MTOK) / 1_000_000, 4)


In [ ]:
# [c10] load_source
# 只讀：A 節讀 v_meeting_sb_reconcile_q；B 節讀 v_meeting_items_q、v_meeting_mail_index、清洗表、隔離表、鏡頭表（只取 item_key / dup_key）、
# 參與者表（只取 token / side / company，不取姓名）。都是 View 或已去識別的欄。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


u = cfg["unit"]
need = {"A": ["v_meeting_sb_reconcile_q"],
        "B": ["v_meeting_items_q", "v_meeting_mail_index", f"s_meeting_record_clean_{u}", f"g_meeting_items_sem_{u}"]}
for sec in cfg["sections"]:
    for t in need[sec]:
        assert spark.catalog.tableExists(fq(t)), f"{fq(t)} 不存在：A 節要 004 ＋ T3 / T4；B 節要 002 / 003 ＋ W1 ＋ Stage A"
fq_skipped, fq_part = fq(f"s_meeting_record_skipped_{u}"), fq(f"s_meeting_participants_{u}")
has_skipped, has_part = spark.catalog.tableExists(fq_skipped), spark.catalog.tableExists(fq_part)

df_rec_q = spark.table(fq("v_meeting_sb_reconcile_q")) if "A" in cfg["sections"] else None
if "B" in cfg["sections"]:
    df_items_m = attach_months(spark.table(fq("v_meeting_items_q")), spark.table(fq("v_meeting_mail_index")),
                               spark.table(fq(f"g_meeting_items_sem_{u}")))
    df_mail_idx, df_clean = spark.table(fq("v_meeting_mail_index")), spark.table(fq(f"s_meeting_record_clean_{u}"))
    df_skipped = spark.table(fq_skipped) if has_skipped else None
    df_part = spark.table(fq_part).select(*PART_COLS) if has_part else None
print(f"sections {cfg['sections']} | months {cfg['months']} | skipped table {has_skipped} | participants {has_part}")


In [ ]:
# [c11] run
# 純函式串起來：A 節一次（多月一份）；B 節每月：漏斗 → 分組 → LLM 摘要（ThreadPool；條目多的組先、受 max_llm_groups / min_items_for_llm 閘門）→ 卡片。
# 最後組總覽卡放最前。每月 collect 上限 max_rows（超過印 WARN，只列前 max_rows 條）。
from datetime import datetime

md_parts: list[str] = []
cards: list[str] = []
nav: list[tuple[str, str]] = [("overview", "總覽")]
stats: dict = {}
badge = 2  # 1 留給總覽
fq_rec_table = fq(f"g_meeting_sb_reconcile_{cfg['unit']}")
usage_tot = {"input": 0, "output": 0, "cache_read": 0, "cache_write": 0, "calls": 0, "retries": 0, "failed": 0}
latencies: list[float] = []
sb_summary: dict = {}
b_profiles: dict = {}


def _account(r: dict) -> None:
    usage_tot["calls"] += 1
    for k in ("input", "output", "cache_read", "cache_write"):
        usage_tot[k] += (r.get("usage") or {}).get(k, 0)
    latencies.append(r.get("latency_s") or 0.0)


def ask_summary(prompt: str) -> dict:
    """一次呼叫（JSON 壞再問一次）→ postprocess_summary 的 dict。"""
    messages = [{"role": "user", "content": prompt}]
    r1 = call_llm(messages, SYSTEM_PROMPT_PM, cfg["model"], cfg["prompt_cache"])
    _account(r1)
    if r1["error"] or r1["stop_reason"] == "max_tokens":
        return postprocess_summary(None, r1["error"] or "output truncated")
    obj, perr = parse_llm_json(r1["text"])
    if obj is None:
        usage_tot["retries"] += 1
        messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"}, {"role": "user", "content": retry_user_prompt(perr)}]
        r2 = call_llm(messages, SYSTEM_PROMPT_PM, cfg["model"], cfg["prompt_cache"])
        _account(r2)
        if r2["error"] or r2["stop_reason"] == "max_tokens":
            return postprocess_summary(None, r2["error"] or "truncated (retry)")
        obj, perr = parse_llm_json(r2["text"])
    return postprocess_summary(obj, perr)


def summarize_groups(month: str, groups: "OrderedDict", labels: dict) -> dict:
    """{(customer, app): {"summary", "watch", "error" | "skipped"}}；條目多的組先；閘門內的才呼叫。"""
    out: dict = {}
    todo = []
    for (customer, app), rows in groups.items():  # groups 已按條目數降冪
        if not cfg["llm_summary"]:
            out[(customer, app)] = {"skipped": "未呼叫 LLM（llm_summary = false）"}
        elif len(rows) < cfg["min_items_for_llm"]:
            out[(customer, app)] = {"skipped": f"條目少於 {cfg['min_items_for_llm']}，只列代表句"}
        elif cfg["max_llm_groups"] and len(todo) >= cfg["max_llm_groups"]:
            out[(customer, app)] = {"skipped": f"超出 max_llm_groups = {cfg['max_llm_groups']}，只列代表句"}
        else:
            todo.append(((customer, app), build_group_prompt(month, customer, app, rows, labels, basis=cfg["month_basis"])))
    if todo:
        with ThreadPoolExecutor(max_workers=cfg["concurrency"]) as ex:
            futs = {ex.submit(ask_summary, p): k for k, p in todo}
            for fut in as_completed(futs):
                res = fut.result()
                out[futs[fut]] = res
                if res.get("error"):
                    usage_tot["failed"] += 1
    return out


if "A" in cfg["sections"]:
    cells = [r.asDict() for r in select_sb_cells(df_rec_q, cfg["months"]).collect()]
    sb_summary = summarize_sb(cells)
    md_parts.append(render_sb_md(cells, cfg["months"], max_text=cfg["max_text"]))
    a_cards, badge = render_sb_html(cells, cfg["months"], max_text=cfg["max_text"], badge_start=badge)
    cards += a_cards
    nav += [(f"a-{m}", f"A｜{m} 對照") for m in cfg["months"]]
    stats["A_cells"] = dict(Counter(c["brief_month"] for c in cells))

if "B" in cfg["sections"]:
    basis = cfg["month_basis"]
    fm = {r["month"]: r.asDict() for r in mail_funnel(df_mail_idx, cfg["months"]).collect()}
    fs = {r["month"]: r.asDict() for r in skipped_funnel(df_skipped, cfg["months"]).collect()} if df_skipped is not None else {}
    fc = {r["month"]: r.asDict() for r in clean_funnel(df_clean, cfg["months"]).collect()}
    fi = {r["month"]: r.asDict() for r in items_funnel(df_items_m, cfg["months"], basis=basis).collect()}
    no_date = items_no_date(df_items_m).first()["items_no_date"] if basis == "meeting" else 0
    md_parts.append("# B. 當月會議紀錄彙整\n")
    t0 = time.time()
    for month in cfg["months"]:
        funnel = {**fm.get(month, {}), **fs.get(month, {}), **fc.get(month, {}), **fi.get(month, {}), "items_no_date": no_date}
        rows = [r.asDict() for r in select_month_items(df_items_m, month, basis=basis, include_dups=cfg["include_dups"])
                .limit(cfg["max_rows"] + 1).collect()]
        if len(rows) > cfg["max_rows"]:
            print(f"WARN {month}：條目超過 max_rows {cfg['max_rows']}，只列前 {cfg['max_rows']} 條；調大 max_rows 或分月跑")
            rows = rows[: cfg["max_rows"]]
        pks = sorted({r["pk_seq"] for r in rows})
        labels = token_labels([r.asDict() for r in df_part.where(F.col("pk_seq").isin(*pks)).collect()]) if (df_part is not None and pks) else {}
        groups = group_items(rows, include_dups=cfg["include_dups"])
        b_profiles[month] = {k: group_profile(v) for k, v in groups.items()}
        summaries = summarize_groups(month, groups, labels)
        kw = dict(basis=basis, include_dups=cfg["include_dups"], include_quote=cfg["include_quote"], max_per_group=cfg["max_per_group"], summaries=summaries)
        md_parts.append(render_monthly_md(month, funnel, rows, labels, **kw))
        cards.append(render_monthly_html(month, funnel, rows, labels, badge=badge, **kw))
        nav.append((f"b-{month}", f"B｜{month} 會議紀錄"))
        badge += 1
        stats[f"B_{month}"] = {k: funnel.get(k) for k in ("mails", "items_valid", "items_canonical")} | {
            "groups": len(groups), "llm_ok": sum(1 for s in summaries.values() if s.get("summary")),
            "llm_failed": sum(1 for s in summaries.values() if s.get("error")), "skipped": sum(1 for s in summaries.values() if s.get("skipped"))}
    stats["llm"] = {**usage_tot, "est_cost_usd": est_cost_usd(usage_tot), "latency_max_s": max(latencies) if latencies else None,
                    "elapsed_s": round(time.time() - t0, 1)}

overview = build_overview(cfg["months"], sb_summary, b_profiles)
cards.insert(0, render_overview_html(overview, badge=1, has_a="A" in cfg["sections"], has_b="B" in cfg["sections"]))
md_parts.insert(0, render_overview_md(overview))

generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")
title = f"會議紀錄 × Sales Brief｜PM 審核包 {'、'.join(cfg['months'])}"
meta = [f"產生 {generated_at}", f"節：{' + '.join(cfg['sections'])}", f"B 節月份口徑：{cfg['month_basis']}",
        f"摘要 prompt {PROMPT_VERSION}（{cfg['model']}）" if cfg["llm_summary"] else "無 LLM 摘要", "內部版：含 S2、人名以陣營顯示"]
warn = "資料缺口：約三成信件 ETL 截斷無正文、附件不讀、客戶欄約四成空值；會議內容是各方陳述，不是已驗證數據。"
md = "\n".join(md_parts)
html = render_page_html(title, meta, cards, warn=warn, nav=nav)
print(f"md {len(md):,} 字 | html {len(html):,} 字 | {json.dumps(stats, ensure_ascii=False, default=str)}")


In [ ]:
# [c12] print_output
# output = html：先 displayHTML 預覽，再把整份 HTML 原始碼印出來 → 全選複製、存成 .html（UTF-8）給 PM 用瀏覽器開（純閱讀，無 JS）。
# output = md：分段印 markdown。both：兩個都印。不落地、不寫表。
if cfg["output"] in ("html", "both"):
    displayHTML(html)
    print("<!-- ===== 下面整段複製，存成 pm_pack_<months>.html（UTF-8） ===== -->")
    print(html)
if cfg["output"] in ("md", "both"):
    for i, part in enumerate(md_parts, 1):
        print(f"\n<!-- part {i}/{len(md_parts)} -->\n")
        print(part)
